In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## Schritt 1: Definieren Sie Pydantic-Modelle für strukturierte Ausgaben

Diese Modelle definieren das **Schema**, das Agenten zurückgeben werden. Die Verwendung von `response_format` mit Pydantic gewährleistet:
- ✅ Typensichere Datenauswertung
- ✅ Automatische Validierung
- ✅ Keine Parsing-Fehler bei Freitextantworten
- ✅ Einfache bedingte Weiterleitung basierend auf Feldern


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Schritt 2: Erstellen des Hotelbuchungs-Tools

Dieses Tool wird vom **availability_agent** aufgerufen, um zu prüfen, ob Zimmer verfügbar sind. Wir verwenden den Dekorator `@ai_function`, um:
- Eine Python-Funktion in ein KI-aufrufbares Tool zu verwandeln
- Automatisch ein JSON-Schema für das LLM zu erzeugen
- Parametervalidierung zu übernehmen
- Automatische Aufrufe durch Agenten zu ermöglichen

Für diese Demo:
- **Stockholm, Seattle, Tokio, London, Amsterdam** → Haben Zimmer ✅
- **Alle anderen Städte** → Keine Zimmer ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Schritt 3: Bedingungsfunktionen für die Weiterleitung definieren

Diese Funktionen überprüfen die Antwort des Agenten und bestimmen, welchen Pfad im Workflow eingeschlagen wird.

**Wichtiges Muster:**
1. Prüfen, ob die Nachricht `AgentExecutorResponse` ist
2. Die strukturierte Ausgabe (Pydantic-Modell) parsen
3. `True` oder `False` zurückgeben, um die Weiterleitung zu steuern

Der Workflow bewertet diese Bedingungen an **Kanten**, um zu entscheiden, welchen Executor er als Nächstes aufruft.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## Schritt 4: Benutzerdefinierten Anzeige-Executor erstellen

Executors sind Workflow-Komponenten, die Transformationen oder Nebeneffekte ausführen. Wir verwenden den `@executor` Dekorator, um einen benutzerdefinierten Executor zu erstellen, der das Endergebnis anzeigt.

**Wichtige Konzepte:**
- `@executor(id="...")` - Registriert eine Funktion als Workflow-Executor
- `WorkflowContext[Never, str]` - Typ-Hinweise für Eingabe/Ausgabe
- `ctx.yield_output(...)` - Gibt das endgültige Workflow-Ergebnis aus


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## Schritt 5: Umgebungsvariablen laden

Konfigurieren Sie den LLM-Client. Dieses Beispiel funktioniert mit:
- **GitHub-Modelle** (kostenlose Stufe mit GitHub-Token)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Schritt 6: Erstellen von KI-Agenten mit strukturierten Ausgaben

Wir erstellen **drei spezialisierte Agenten**, die jeweils in einem `AgentExecutor` eingehüllt sind:

1. **availability_agent** – Überprüft die Hotelverfügbarkeit mit dem Tool
2. **alternative_agent** – Schlägt alternative Städte vor (wenn keine Zimmer verfügbar sind)
3. **booking_agent** – Ermutigt zur Buchung (wenn Zimmer verfügbar sind)

**Hauptmerkmale:**
- `tools=[hotel_booking]` – Stellt dem Agenten das Tool zur Verfügung
- `response_format=PydanticModel` – Erzwingt strukturierte JSON-Ausgabe
- `AgentExecutor(..., id="...")` – Verpackt den Agenten zur Workflow-Nutzung


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Schritt 7: Erstelle den Workflow mit bedingten Kanten

Nun verwenden wir `WorkflowBuilder`, um den Graphen mit bedingter Steuerung zu erstellen:

**Workflowstruktur:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**Wichtige Methoden:**
- `.set_start_executor(...)` - Legt den Einstiegspunkt fest
- `.add_edge(from, to, condition=...)` - Fügt eine bedingte Kante hinzu
- `.build()` - Schließt den Workflow ab


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## Schritt 8: Testfall 1 ausführen - Stadt OHNE Verfügbarkeit (Paris)

Lassen Sie uns den Pfad **keine Verfügbarkeit** testen, indem wir Hotels in Paris anfragen (das in unserer Simulation keine Zimmer hat).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Schritt 9: Führe Testfall 2 aus - Stadt MIT Verfügbarkeit (Stockholm)

Testen wir nun den **Verfügbarkeits**-Pfad, indem wir Hotels in Stockholm anfragen (die in unserer Simulation Zimmer haben).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## Wichtige Erkenntnisse und nächste Schritte

### ✅ Was Sie gelernt haben:

1. **WorkflowBuilder-Muster**
   - Verwenden Sie `.set_start_executor()`, um den Einstiegspunkt festzulegen
   - Verwenden Sie `.add_edge(from, to, condition=...)` für bedingtes Routing
   - Rufen Sie `.build()` auf, um den Workflow abzuschließen

2. **Bedingtes Routing**
   - Bedingungsfunktionen prüfen `AgentExecutorResponse`
   - Analysieren strukturierte Ausgaben, um Routing-Entscheidungen zu treffen
   - Geben Sie `True` zurück, um eine Kante zu aktivieren, `False`, um sie zu überspringen

3. **Tool-Integration**
   - Verwenden Sie `@ai_function`, um Python-Funktionen in KI-Tools umzuwandeln
   - Agenten rufen Tools bei Bedarf automatisch auf
   - Tools geben JSON zurück, das Agenten analysieren können

4. **Strukturierte Ausgaben**
   - Verwenden Sie Pydantic-Modelle für typsichere Datenauszüge
   - Setzen Sie `response_format=MyModel` bei der Erstellung von Agenten
   - Analysieren Sie Antworten mit `Model.model_validate_json()`

5. **Benutzerdefinierte Executor**
   - Verwenden Sie `@executor(id="...")`, um Workflow-Komponenten zu erstellen
   - Executor können Daten transformieren oder Nebeneffekte ausführen
   - Verwenden Sie `ctx.yield_output()`, um Workflow-Ergebnisse zu erzeugen

### 🚀 Anwendungsbeispiele aus der Praxis:

- **Reisebuchung**: Verfügbarkeit prüfen, Alternativen vorschlagen, Optionen vergleichen
- **Kundendienst**: Routing basierend auf Problemtyp, Stimmung, Priorität
- **E-Commerce**: Lagerbestand prüfen, Alternativen vorschlagen, Bestellungen verarbeiten
- **Inhaltsmoderation**: Routing basierend auf Toxizitätswerten, Nutzer-Meldungen
- **Genehmigungsworkflows**: Routing basierend auf Betrag, Nutzerrolle, Risikoniveau
- **Mehrstufige Verarbeitung**: Routing basierend auf Datenqualität, Vollständigkeit

### 📚 Nächste Schritte:

- Komplexere Bedingungen hinzufügen (mehrere Kriterien)
- Schleifen mit Workflow-Zustandsverwaltung implementieren
- Teil-Workflows für wiederverwendbare Komponenten hinzufügen
- Integration mit echten APIs (Hotelbuchung, Inventarsysteme)
- Fehlerbehandlung und Ausweichpfade hinzufügen
- Visualisierung von Workflows mit den integrierten Visualisierungstools


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
